**Esercizio 1**. Si modifichi il metodo `fit` della classe `LogisticRegressionGD` in modo da implementare la versione stocastica dell'algoritmo della discesa del gradiente.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

class LogisticRegressionGD(object):
    """Logistic Regression Classifier using gradient descent.

    Parameters
    ------------
    eta : float
      Learning rate (between 0.0 and 1.0)
    n_iter : int
    
      Passes over the training dataset.
    random_state : int
      Random number generator seed for random weight
      initialization.


    Attributes
    -----------
    w_ : 1d-array
      Weights after fitting.
    cost_ : list
      Logistic cost function value in each epoch.

    """
    def __init__(self, eta=0.05, n_iter=50, random_state=1, batch_size=None):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state
        self.batch_size = batch_size
    def fit(self, X, y):
        """ Fit training data.

        Parameters
        ----------
        X : {array-like}, shape = [n_examples, n_features]
          Training vectors, where n_examples is the number of examples and
          n_features is the number of features.
        y : array-like, shape = [n_examples]
          Target values.

        Returns
        -------
        self : object

        """
        rgen = np.random.RandomState(self.random_state)
        self.w_ = rgen.normal(loc=0.0, scale=0.01, size=1 + X.shape[1])
        self.cost_ = []
        for i in range(self.n_iter):
            X, y = self._shuffle(X, y)
            epoch_cost = []
            for k in range(0, X.shape[0], self.batch_size):
                Xi = X[k : k + self.batch_size]
                yi = y[k : k + self.batch_size]
                output = self.activation(self.net_input(Xi))    
                error = yi - output
                self.w_[1:] += self.eta * Xi.dot(error) 
                self.w_[0] += self.eta * error
                output = np.clip(output, 1e-15, 1 - 1e-15)
                c = -(yi * np.log(output) + (1 - yi) * np.log(1 - output))
                epoch_cost.append(c)
            
            # Il costo dell'epoca è la media dei costi dei singoli esempi
            self.cost_.append(sum(epoch_cost)/len(y))
        
        return self
    
    def net_input(self, X):
        """Calculate net input"""
        return np.dot(X, self.w_[1:]) + self.w_[0]

    def activation(self, z):
        """Compute logistic sigmoid activation"""
        #return 1. / (1. + np.exp(-np.clip(z, -250, 250)))
        return 1. / (1. + np.exp(-z))

    def predict(self, X):
        """Return class label after unit step"""
        return np.where(self.net_input(X) >= 0.0, 1, 0)
        # equivalent to:
        # return np.where(self.activation(self.net_input(X)) >= 0.5, 1, 0)
    def _shuffle(self, X, y):
        r = np.random.permutation(len(y))
        return X[r], y[r]


In [34]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X,y = make_classification(n_samples=1000, n_features=10,
                        n_informative=5, n_classes=2, random_state=0, shuffle=False)

# Le informative sono le prime n_informative, poi seguono le altre

X = X [:,:5]

X_train, X_test, y_train, y_test = train_test_split(X, y)

mu, std = X_train.mean(0), X_train.std(0)
X_train_std = (X_train-mu)/std
X_test_std = (X_test-mu)/std

X_train_cur, X_test_cur = X_train, X_test

lgrg = LogisticRegressionGD(n_iter=1000, eta=0.00011, random_state=1)
lgrg.fit(X_train_cur,y_train)
accuracy = np.mean(lgrg.predict(X_train_cur) == y_train)
print(accuracy)
accuracy = np.mean(lgrg.predict(X_test_cur) == y_test)
print(accuracy)

0.9146666666666666
0.888


**Esercizio 2**. Vogliamo addestrare un modello di classificazione binaria basato su regressione logistica in un contesto online, in cui i campioni di training arrivano mescolati a quelli di test.

Per abilitare l'*apprendimento continuo*, si propone di modificare la classe `LogisticRegression` aggiungendo un metodo `update(x, y)` che riceve in input un campione `x` e la sua etichetta `y`, e aggiorna i pesi del modello.

Si realizzi inoltre un esperimento per valutare come variano le prestazioni del modello all’aumentare del numero di campioni etichettati utilizzati per l’aggiornamento che simuli uno stream di dati con campioni misti (test e training).

In [35]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

class LogisticRegressionGD(object):
    """Logistic Regression Classifier using gradient descent.

    Parameters
    ------------
    eta : float
      Learning rate (between 0.0 and 1.0)
    n_iter : int
    
      Passes over the training dataset.
    random_state : int
      Random number generator seed for random weight
      initialization.


    Attributes
    -----------
    w_ : 1d-array
      Weights after fitting.
    cost_ : list
      Logistic cost function value in each epoch.

    """
    def __init__(self, eta=0.05, n_iter=50, random_state=1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state
        self.cost_ = []
        self.w_ = None
    def fit(self, X, y):
        """ Fit training data.

        Parameters
        ----------
        X : {array-like}, shape = [n_examples, n_features]
          Training vectors, where n_examples is the number of examples and
          n_features is the number of features.
        y : array-like, shape = [n_examples]
          Target values.

        Returns
        -------
        self : object

        """
        rgen = np.random.RandomState(self.random_state)
        self.w_ = rgen.normal(loc=0.0, scale=0.01, size=1 + X.shape[1])
        cost = []
        for i in range(self.n_iter):
            X, y = self._shuffle(X, y)
            epoch_cost = []
            for xi, target in zip(X, y):
                output = self.activation(self.net_input(xi))    
                error = target - output
                self.w_[1:] += self.eta * xi.dot(error) 
                self.w_[0] += self.eta * error
                output = np.clip(output, 1e-15, 1 - 1e-15)
                c = -(target * np.log(output) + (1 - target) * np.log(1 - output))
                epoch_cost.append(c)
            
            # Il costo dell'epoca è la media dei costi dei singoli esempi
            self.cost_.append(sum(epoch_cost)/len(y))
        
        return self
    
    def net_input(self, X):
        """Calculate net input"""
        return np.dot(X, self.w_[1:]) + self.w_[0]

    def activation(self, z):
        """Compute logistic sigmoid activation"""
        #return 1. / (1. + np.exp(-np.clip(z, -250, 250)))
        return 1. / (1. + np.exp(-z))

    def predict(self, X):
        """Return class label after unit step"""
        return np.where(self.net_input(X) >= 0.0, 1, 0)
        # equivalent to:
        # return np.where(self.activation(self.net_input(X)) >= 0.5, 1, 0)
    def _shuffle(self, X, y):
        r = np.random.permutation(len(y))
        return X[r], y[r]
    def update(self, x, y):
        """Update the model weights with a single training example."""
        if self.w_ is None:
            rgen = np.random.RandomState(self.random_state)
            self.w_ = rgen.normal(loc=0.0, scale=0.01, size=1 + x.shape[0])
        output = np.clip(self.activation(self.net_input(x)))
        error = y - output
        self.w_[1:] += self.eta * x.dot(error)
        self.w_[0] += self.eta * error
        self.cost_.append(-(y * np.log(output) + (1 - y) * np.log(1 - output)))

In [36]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Generiamo il dataset totale (1000 campioni, 5 caratteristiche informative)
X_raw, y_raw = make_classification(n_samples=1000, 
                                   n_features=5, 
                                   n_informative=5, 
                                   n_redundant=0, 
                                   n_classes=2, 
                                   random_state=1)

# 2. Standardizzazione (Fondamentale per la Regressione Logistica)
# In un contesto online "puro" dovresti aggiornare mu e std man mano, 
# ma per l'esercizio usiamo quelli iniziali.
mu, std = X_raw.mean(axis=0), X_raw.std(axis=0)
X = (X_raw - mu) / std
y = y_raw

# 3. Mescolamento (Per simulare l'arrivo casuale nello stream)
indices = np.arange(X.shape[0])
np.random.shuffle(indices)
X_stream = X[indices]
y_stream = y[indices]

In [38]:
# Inizializzazione
lgrg_online = LogisticRegressionGD(eta=0.01)

# Liste per tracciare le differenze
acc_test_pre = []   # Accuratezza prima dell'update (Generalizzazione)
acc_train_post = [] # Accuratezza dopo l'update (Apprendimento)
checkpoints = [10, 100, 500, 1000]

print(f"{'Campioni':<10} | {'Acc. Test (Pre)':<15} | {'Acc. Train (Post)':<15} | {'Gap (Overfitting?)'}")
print("-" * 70)

successi_test = 0
successi_train = 0

for i in range(len(X_stream)):
    xi = X_stream[i]
    yi = y_stream[i]
    
    if lgrg_online.w_ is not None:
        # --- PRE-UPDATE (Test) ---
        pred_test = lgrg_online.predict(xi.reshape(1, -1))
        if pred_test == yi:
            successi_test += 1
        
        # --- UPDATE (Learning) ---
        lgrg_online.update(xi, yi)
        
        # --- POST-UPDATE (Train) ---
        pred_train = lgrg_online.predict(xi.reshape(1, -1))
        if pred_train == yi:
            successi_train += 1
            
        # Calcolo medie cumulate
        acc_test_pre.append(successi_test / (i + 1))
        acc_train_post.append(successi_train / (i + 1))

    else:
        # Inizializzazione al primo campione
        lgrg_online.update(xi, yi)

    # Stampa tabella ai checkpoint
    if (i + 1) in checkpoints:
        t_acc = acc_test_pre[-1]
        tr_acc = acc_train_post[-1]
        gap = tr_acc - t_acc
        print(f"{i+1:<10} | {t_acc:<15.4f} | {tr_acc:<15.4f} | {gap:<15.4f}")

Campioni   | Acc. Test (Pre) | Acc. Train (Post) | Gap (Overfitting?)
----------------------------------------------------------------------
10         | 0.6000          | 0.7000          | 0.1000         
100        | 0.7600          | 0.8300          | 0.0700         
500        | 0.7780          | 0.8020          | 0.0240         
1000       | 0.8010          | 0.8130          | 0.0120         
